# 01 — Data Preparation

This notebook prepares the Greek real-estate transaction dataset for the subsequent stages of the project.

The overall objective of this project is to demonstrate an end-to-end data science workflow using real-world administrative data:

**Data acquisition → Data understanding → Data cleaning → Transformation → Feature engineering → Descriptive analysis → Statistical analysis → Regression & Machine Learning**

The focus of this notebook is data preparation rather than modelling. The goal is to construct a reliable analytical dataset that can be used in the subsequent stages of the project.

An important part of the preprocessing was understanding the structure and meaning of the source data. The dataset was not accompanied by a comprehensive public data dictionary or detailed documentation for all variables used in the analysis. Therefore, some aspects of the dataset had to be reconstructed through exploratory analysis, examination of value distributions and relationships, and plausibility checks.

Where assumptions or project-specific decisions were necessary, they are documented explicitly rather than presented as official definitions from the data source.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.float_format', '{:.2f}'.format)

## 2 — Loading the raw dataset

The data originate from the Greek Real Estate Transactions Valuation Register (Μητρώο Αξιών Μεταβιβάσεων Ακινήτων).

For this project, annual datasets covering 2017–2026 were manually downloaded and combined into a single dataset.

The resulting `raw_data.csv` is used as the raw input for the preprocessing pipeline.

*Source: https://webapps.gsis.gr/dsae2/trxregistry/faces/pages/mainmenu/edit.xhtml;jsessionid=cWxMz3u0_eDktDjnO09pMLFSq5u6vbFhbp1VMfLRVtvpKZECMFTP!297911023*

In [ ]:
df = pd.read_csv("../data/raw_data.csv")

## 3 — Initial inspection

Before applying any cleaning rules, the structure and contents of the raw dataset are inspected.

Because the source did not provide a comprehensive data dictionary, this initial exploration is particularly important for understanding the available variables and identifying potential data-quality issues.

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

In [ ]:
df.describe(include='all').T

In [ ]:
df.isna().sum().sort_values(ascending=False)

## 4 — Initial duplicate check

Exact duplicate rows are removed from the integrated raw dataset.

A second duplicate check will also be performed at the end of the preprocessing pipeline. This is useful because subsequent filtering, column removal, and transformations can cause previously distinct rows to become identical in the retained feature space.

In [ ]:
print(f"Duplicate rows before removal: {df.duplicated().sum()}")

df = df.drop_duplicates().copy()

print(f"Duplicate rows after removal: {df.duplicated().sum()}")

## 5 — Defining the residential modelling population

The raw register contains several different categories of real estate, including land, commercial properties, parking spaces, storage areas, and other property types.

The objective of this project is to investigate and eventually model **residential property transaction prices**.

Therefore, the modelling population is restricted to:

- Apartments
- Detached houses

This is a project-specific population definition designed to make the subsequent analysis more coherent.

In [ ]:
df = df[
    (df['Κατηγορία Ακινήτου'] == 'Κατοικία ή διαμέρισμα πλήν μονοκατοικίας') |
    (df['Κατηγορία Ακινήτου'] == 'Μονοκατοικία')
].copy()

In [ ]:
df['Κατηγορία Ακινήτου'].value_counts()

## 6 — Restricting the ownership definition

The source data contain different types of property rights and ownership shares.

To make the transaction-price target more comparable across observations, the analysis focuses on observations representing **full ownership of the building**.

In [ ]:
df = df[
    df['Είδος Εμπράγματου δικαιώματος Κτίσματος'] == 'Πλήρης Κυριότητα'
].copy()

In [ ]:
df = df[
    df['Ποσοστό Συνιδιοκτησίας Κτίσματος'] == 100
].copy()

## 7 — Excluding special property conditions

The dataset contains observations associated with special property conditions.

For the main modelling population, these observations are excluded because such properties may follow different pricing mechanisms from ordinary residential properties.

This is a modelling decision rather than a claim that these transactions are invalid.

In [ ]:
df = df[df['Ειδικές Συνθήκες Ακινήτου'].isna()].copy()

## 8 — Minimum main-area requirement

A minimum main-area threshold of **20 m²** is applied.

This is intended to exclude very small observations that may be atypical for the residential-property population being modelled and could produce disproportionately high price-per-square-metre values.

The threshold is a project-specific data-quality criterion.

In [ ]:
df = df[
    df['Eπιφάνεια Κύριων Χώρων (σε τ.μ.)'] >= 20
].copy()

## 9 — Removing variables not required for the current analysis

Several source variables describe land ownership or other characteristics that are not required for the current residential-property modelling population.

These variables are removed from the working dataframe.

The available municipal-district field is retained because geographic information may be useful in later analysis and modelling.

In [ ]:
df = df.drop(columns=[
    'Συνολική Επιφάνεια Κτισμάτων στο οικόπεδο',
    'Ποσοστό Συνιδιοκτησίας Οικοπέδου',
    'Είδος Εμπράγματου δικαιώματος Οικοπέδου',
    'Επιφάνεια Οικοπέδου (σε τ.μ.)',
    'Ειδικές Συνθήκες Ακινήτου',
    'Ποσοστό Συνιδιοκτησίας Κτίσματος',
    'Είδος Εμπράγματου δικαιώματος Κτίσματος',
    'Νομαρχία',
    'Δήμος Καλλικράτη'
])

## 10 — Handling missing auxiliary-area values

The auxiliary-area variable contains missing values.

For this project, missing auxiliary-area values are treated as **0 m²**.

This is an explicit project assumption. The source data did not provide sufficient documentation to establish definitively why these observations were missing, so this interpretation is documented rather than treated as an official source definition.

In [ ]:
df.loc[
    df['Επιφάνεια Βοηθητικών Χώρων (σε τ.μ.)'].isna(),
    'Επιφάνεια Βοηθητικών Χώρων (σε τ.μ.)'
] = 0

In [ ]:
df['Επιφάνεια Βοηθητικών Χώρων (σε τ.μ.)'].isna().sum()

## 11 — Constructing an objective-value proxy

The dataset contains a zone price and the main area of each property.

A simplified benchmark is constructed as:

**main area × zone price**

This variable is called `objective_value_proxy` because it is **not assumed to be the official objective property value** calculated by the Greek property valuation system.

The official valuation methodology may incorporate additional coefficients and property characteristics. The variable constructed here is therefore used only as a simplified benchmark for data-quality analysis.

In [ ]:
df['objective_value_proxy'] = (
    df['Eπιφάνεια Κύριων Χώρων (σε τ.μ.)']
    * df['Tιμή Ζώνης']
)

## 12 — Transaction-price to proxy-value filtering

The relationship between the recorded transaction price and the constructed objective-value proxy was investigated as part of the data-quality assessment.

Several candidate thresholds were considered.

A threshold of **60%** was ultimately selected for the modelling population. Lower thresholds retained observations with comparatively implausibly low transaction prices relative to the constructed benchmark, while more restrictive thresholds removed a substantially larger portion of the available observations without providing a sufficiently clear improvement in the lower tail.

This threshold is a **project-specific modelling criterion**. It is not an official rule of the source registry and observations below the threshold are not necessarily invalid transactions.

In [ ]:
df = df[
    df['Τίμημα Δικαιώματος'] >=
    (df['objective_value_proxy'] * 0.60)
].copy()

## 13 — Price per square metre

Price per square metre is calculated for descriptive analysis and later data-quality investigation.

It is important to note that this variable is **derived from the target variable**:

**price per m² = transaction price / main area**

Therefore, although it is useful for exploratory analysis, it must **not** be used as a predictor in a model whose target is transaction price, as doing so would introduce target leakage.

In [ ]:
df['price_per_sqm'] = (
    df['Τίμημα Δικαιώματος']
    / df['Eπιφάνεια Κύριων Χώρων (σε τ.μ.)']
)

## 14 — Construction-year quality check

The construction-year variable contains a small number of extremely low values.

These observations were investigated individually. Values below 1700 were considered implausible for the modelling dataset based on their relationship with the corresponding transaction dates and other property characteristics.

Later historical construction years were retained where the associated observations appeared plausible.

This is a data-quality decision based on inspection rather than a claim that historically old buildings are inherently invalid.

In [ ]:
df = df[
    df['Έτος Κατασκευής'] >= 1700
].copy()

## 15 — Floor transformation and quality filtering

The source dataset uses a textual indicator (`Υ`) for basement-level properties.

This value is converted to `-1` so that the floor variable can be represented numerically.

The distribution was then investigated for unusually high floor values. A small number of observations had values above 8 and were treated as a potentially unreliable extreme upper tail for this project.

The upper bound is therefore set at 8.

This does **not** imply that buildings above eight floors cannot exist. The decision concerns the reliability of the recorded values within this particular dataset.

In [ ]:
df.loc[df['Όροφος'] == 'Υ', 'Όροφος'] = '-1'

df['Όροφος'] = df['Όροφος'].astype(int)

In [ ]:
df['Όροφος'].value_counts().sort_index()

In [ ]:
df = df[df['Όροφος'] <= 8].copy()

In [ ]:
df['Όροφος'].min(), df['Όροφος'].max()

## 16 — Renaming variables

The remaining source variables are renamed into English to make the dataset easier to use throughout the rest of the project.

In [ ]:
df = df.rename(columns={
    'Δημοτικό ή Κοινοτικό Διαμέρισμα': 'municipal_district',
    'Ένδειξη ΑΠΑΑ': 'apaa_status',
    'Κατηγορία Ακινήτου': 'property_type',
    'Πλήθος Προσόψεων': 'number_of_facades',
    'Tιμή Ζώνης': 'zone_price_eur_per_m2',
    'Eπιφάνεια Κύριων Χώρων (σε τ.μ.)': 'main_area_m2',
    'Επιφάνεια Βοηθητικών Χώρων (σε τ.μ.)': 'auxiliary_area_m2',
    'Έτος Κατασκευής': 'construction_year',
    'Όροφος': 'floor',
    'Ημερομηνία Συμβολαίου': 'contract_date',
    'Τίμημα Δικαιώματος': 'transaction_price_eur',
    'objective_value_proxy': 'objective_value_proxy',
    'price_per_sqm': 'price_per_m2_eur'
})

## 17 — Transforming categorical variables

The original categorical labels are translated into English.

A binary indicator is also created for later statistical and machine-learning models.

In [ ]:
df['apaa_status'] = df['apaa_status'].replace({
    'Εντός ΑΠΑΑ': 'Within APPA',
    'Εκτός ΑΠΑΑ': 'Outside APPA'
})

In [ ]:
df['within_appa'] = (
    df['apaa_status'] == 'Within APPA'
).astype(int)

In [ ]:
df['property_type'] = df['property_type'].replace({
    'Κατοικία ή διαμέρισμα πλήν μονοκατοικίας': 'Apartment',
    'Μονοκατοικία': 'Detached house'
})

In [ ]:
df['is_apartment'] = (
    df['property_type'] == 'Apartment'
).astype(int)

## 18 — Date conversion and temporal features

The contract date is converted to a pandas datetime variable.

Temporal features are then extracted so that later analysis can investigate changes in transaction prices over time and potential seasonal patterns.

In [ ]:
df['contract_date'] = pd.to_datetime(df['contract_date'])

In [ ]:
df['contract_year'] = df['contract_date'].dt.year

df['contract_month'] = df['contract_date'].dt.month

df['contract_day'] = df['contract_date'].dt.day

df['contract_day_of_week'] = df['contract_date'].dt.day_of_week

df['contract_quarter'] = df['contract_date'].dt.quarter

## 19 — Final duplicate check

A final duplicate check is performed after the complete cleaning and transformation pipeline.

This is necessary because removing columns and transforming variables can cause records that were previously distinct to become identical in the final feature space.

Five exact duplicate observations were identified at this stage and removed.

In [ ]:
print(f"Duplicate rows before final removal: {df.duplicated().sum()}")

df = df.drop_duplicates().copy()

print(f"Duplicate rows after final removal: {df.duplicated().sum()}")

## 20 — Resetting the index

The original dataframe index is no longer meaningful after the filtering operations.

The index is therefore reset before the cleaned dataset is exported.

In [ ]:
df = df.reset_index(drop=True)

## 21 — Final data-quality validation

Before exporting the cleaned dataset, the final structure and key data-quality properties are checked.

The validation focuses on:

- missing values,
- duplicate observations,
- numerical ranges,
- date ranges,
- and consistency of engineered variables.

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.isna().sum()

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
print(f"Floor range: {df['floor'].min()} to {df['floor'].max()}")

In [ ]:
print(
    f"Construction year range: "
    f"{df['construction_year'].min()} to "
    f"{df['construction_year'].max()}"
)

In [ ]:
print(
    f"Contract date range: "
    f"{df['contract_date'].min()} to "
    f"{df['contract_date'].max()}"
)

### Consistency checks

The two variables constructed during preprocessing are independently validated against their definitions.

The maximum absolute difference should be `0.0` in both cases.

In [ ]:
price_per_m2_error = (
    df['price_per_m2_eur']
    - df['transaction_price_eur'] / df['main_area_m2']
).abs().max()

print(f"Maximum price-per-m² calculation error: {price_per_m2_error}")

In [ ]:
proxy_error = (
    df['objective_value_proxy']
    - df['main_area_m2'] * df['zone_price_eur_per_m2']
).abs().max()

print(f"Maximum objective-value-proxy calculation error: {proxy_error}")

## 22 — Final analytical dataset

After the cleaning, transformation, and validation process, the dataset contains the residential property transactions that satisfy the project's modelling criteria.

The resulting dataset contains:

- geographic information,
- APPA status,
- property type,
- number of facades,
- zone price,
- main and auxiliary area,
- construction year,
- floor,
- contract date,
- transaction price,
- a constructed objective-value proxy,
- price per square metre,
- binary APPA and property-type indicators,
- and temporal features.

The dataset is now ready for the subsequent descriptive, statistical, regression, and machine-learning stages.

## 23 — Exporting the cleaned dataset

Two versions of the cleaned dataset are exported.

**CSV** provides a widely compatible and human-readable format.

**Parquet** is used as the preferred analytical format because it preserves column data types, including the `contract_date` datetime type, and is generally more efficient for repeated analytical workflows.

In [ ]:
df.to_csv("../data/cleaned_data.csv", index=False)

df.to_parquet("../data/cleaned_data.parquet", index=False)

## 24 — Final verification

The exported analytical dataset is now frozen and will serve as the input dataset for the following notebooks.

The next stage of the project will focus on descriptive analysis rather than further preprocessing.

In [ ]:
print(f"Final dataset shape: {df.shape}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Total missing values: {df.isna().sum().sum()}")